### Data Preparation

In [ ]:
from nexcsi import decoder
import numpy as np

device = "raspberrypi"


def extract_time_and_csi(samples: np.ndarray) -> np.ndarray:
    timestamps = samples['ts_sec'].astype(np.uint64) * 1_000_000 + samples['ts_usec'].astype(np.uint64)
    csi_unpacked = decoder(device).unpack(samples['csi'])

    # remove nulls and pilots
    remove_indices = [0, 1, 2, 3, 11, 25, 32, 39, 53, 60, 61, 62, 63]
    csi_no_nulls = np.delete(csi_unpacked, remove_indices, axis=1)

    out = np.empty(
        len(samples),
        dtype=[
            ('timestamp', np.uint64),
            ('csi', csi_no_nulls.dtype, csi_no_nulls.shape[1:])
        ]
    )
    out['timestamp'] = timestamps
    out['csi'] = csi_no_nulls
    return out


def extract_csi_images(samples: np.ndarray, window_size :int = 10, ms_diff_tolerance: float = 110, sliding: bool = True) -> np.ndarray:
    if sliding is False:
        step_size = window_size
    else:
        step_size = 1

    csi_images = []
    for i in range(0, samples.shape[0] - window_size + 1, step_size):
        window = samples[i:i+window_size]
        time_diffs_ms = np.diff(window['timestamp']) / 1_000
        if np.all(time_diffs_ms <= ms_diff_tolerance):
            csi_images.append(window['csi'])
    
    return np.array(csi_images)


def csi_image_shuffle_subcarriers(csi_images: np.ndarray, x: int, fraction: float = 1.0) -> np.ndarray:
    """
    Shuffles the subcarrier axis for a specific fraction of the input images
    and returns a single concatenated array.
    """
    if not (0.0 <= fraction <= 1.0):
        raise ValueError(f"Fraction must be between 0.0 and 1.0. Received: {fraction}")
        
    N, T, S = csi_images.shape
    num_to_shuffle = int(N * fraction)
    
    # If the fraction is 0, there is nothing to shuffle. 
    # Return a copy of the original array so it isn't accidentally mutated later.
    if num_to_shuffle == 0:
        return csi_images.copy()
        
    # Split the input array
    images_to_shuffle = csi_images[:num_to_shuffle]
    untouched_images = csi_images[num_to_shuffle:]
    
    # Apply the vectorization ONLY to the target fraction
    repeated_images = np.repeat(images_to_shuffle, x, axis=0)
    
    random_noise = np.random.rand(num_to_shuffle * x, S)
    rand_idx = np.argsort(random_noise, axis=1)
    rand_idx = rand_idx[:, np.newaxis, :]
    
    shuffled_images = np.take_along_axis(repeated_images, rand_idx, axis=2)
    
    # Concatenate the shuffled images and the untouched images back together
    return np.concatenate((shuffled_images, images_to_shuffle, untouched_images), axis=0)


def image_to_dataset(csi_images: np.ndarray, label: int) -> np.ndarray:
    assert label in (0, 1), "label must be 0 (no motion) or 1 (motion)"

    csi_mag = np.log10(
        np.maximum(np.abs(csi_images).astype(np.float32), np.finfo(np.float32).eps)
    )

    # per-sample min/max over (64, 51), keeping sample axis intact
    csi_log_min = csi_mag.min(axis=(1, 2), keepdims=True)
    csi_log_max = csi_mag.max(axis=(1, 2), keepdims=True)

    csi_log_normalized = (csi_mag - csi_log_min) / (csi_log_max - csi_log_min + 1e-8)
    csi_log_normalized = csi_log_normalized.astype(np.float32)

    dataset = np.empty(
        csi_images.shape[0],
        dtype=[
            ('periodogram', np.float32, csi_images.shape[1:]),
            ('label', np.int8)
        ]
    )

    dataset['periodogram'] = csi_log_normalized
    dataset['label'] = label
    return dataset


def build_dataset(files: list[tuple[str, int, bool]], window_size: int = 32, ms_diff_tolerance: float = 110, sliding: bool = True) -> np.ndarray:
    datasets = []
    for file, label, shuffle in files:
        samples = extract_time_and_csi(decoder(device).read_pcap(file))

        ts = samples['timestamp']
        violations = np.where(ts[1:] < ts[:-1])[0]

        assert violations.size == 0, (
            f"timestamps are not in ascending order; first violation at index {violations[0]}: "
            f"{ts[violations[0]]} -> {ts[violations[0] + 1]}"
        )

        csi_images = extract_csi_images(samples, window_size, ms_diff_tolerance, sliding)
        if csi_images.size == 0:
            print(f"Warning: no valid CSI images extracted from {file}; skipping this file.")
            continue
        if shuffle:
            csi_images = csi_image_shuffle_subcarriers(csi_images, x=2, fraction=0.2)
        dataset = image_to_dataset(csi_images, label)
        datasets.append(dataset)
    
    return np.concatenate(datasets)


In [ ]:
data = build_dataset([
            ('motion-1.pcap', 1,                True),
            ('motion-2.pcap', 1,                True),
            ('motion-3.pcap', 1,                True),

            ('no-motion-1.pcap',  0,            True),
            ('no-motion-2.pcap',  0,            True),
            ('no-motion-3.pcap',  0,            True),
        ])

In [ ]:
np.save("dataset.npy", data)

### Model Training

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models # type: ignore
import numpy as np

2026-03-11 16:05:26.319425: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-11 16:05:26.397071: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-11 16:05:27.916397: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
data = np.load("dataset.npy", allow_pickle=False)
np.random.shuffle(data)
data.shape

(82209,)

In [ ]:
data_size = data.shape[0]

# Split the data into training, validation, and test sets (70% training, 15% validation, 15% test)
train_end = int(data_size * 0.70)
val_end = int(data_size * 0.85)

X_train = data['periodogram'][:train_end]
X_train = X_train[..., np.newaxis]
y_train = data['label'][:train_end]
y_train = y_train[..., np.newaxis]

X_val = data['periodogram'][train_end:val_end]
X_val = X_val[..., np.newaxis]
y_val = data['label'][train_end:val_end]
y_val = y_val[..., np.newaxis]

X_test = data['periodogram'][val_end:]
X_test = X_test[..., np.newaxis]
y_test = data['label'][val_end:]
y_test = y_test[..., np.newaxis]

In [ ]:
# Share of 0s and 1s in y_train
total = y_train.size
count_0 = np.sum(y_train == 0)
count_1 = np.sum(y_train == 1)

print(f"0s: {count_0} ({count_0/total:.2%})")
print(f"1s: {count_1} ({count_1/total:.2%})")

0s: 18047 (31.36%)
1s: 39499 (68.64%)


In [ ]:
X_train[0].shape

(32, 51, 1)

In [ ]:
model = models.Sequential([
    layers.Input(shape=X_train[0].shape),

    layers.Conv2D(8, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(4, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(2, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Dropout(0.5),

    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
])

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=25,
        restore_best_weights=True
    )
]

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=50,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=callbacks
)

In [ ]:
model.evaluate(X_test, y_test)

386/386 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9938 - loss: 0.0200


[0.019985774531960487, 0.9937560558319092]

In [ ]:
X_test.shape

(12332, 32, 51, 1)

In [ ]:
X_test[22].shape

(32, 51, 1)

In [ ]:
np.expand_dims(X_test[22], axis=0).shape

(1, 32, 51, 1)

In [ ]:
X_new = np.expand_dims(X_test[22], axis=0)
X_new.shape

model.predict(X_new)[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step


0.0002134076

In [ ]:
model.save("model.keras")